# Gold reporting views for the dashboard and Genie Agent

Owner: Person 4. This notebook creates documented, read-only views in the Gold schema. The AI/BI dashboard and the Genie Agent read only these views, so both use the same definitions, denominators and units.

Inputs:

- `airbnb_silver.listings` and `airbnb_gold.city_metrics`, produced by the parameterized ETL pipeline (`02_etl_pipeline`).
- `airbnb_gold.predictions`, produced by the price model (`03_price_model`). Model views are created only when this table exists and follows the data contract; otherwise the notebook skips them and reports why.

Definitions applied throughout:

- Price is the nightly advertised price in GBP on the snapshot date. Only rows with `target_valid = true` contribute to price statistics; missing prices are never treated as zero.
- Medians are exact (`percentile(..., 0.5)`) and may differ marginally from approximate medians stored by the ETL metric tables.
- Availability is the number of calendar days available in the next 365 days. It is not occupancy. Values failing ETL range checks are excluded.
- Every aggregate carries its own valid-row denominator.
- Model error is `predicted - actual`. A negative error means the model under-prices the listing.
- Reported model metrics use the test split of training cities and all rows of any non-training (transfer) city.

No host identifiers, names, descriptions or listing URLs are exposed. The views contain no city-specific logic; a new city appears once the ETL has processed it.

## 1. Parameters

In [ ]:
import re

dbutils.widgets.text('catalog', 'workspace')
dbutils.widgets.text('silver_schema', 'airbnb_silver')
dbutils.widgets.text('gold_schema', 'airbnb_gold')
dbutils.widgets.text('training_cities', 'london,bristol')
dbutils.widgets.text('reference_city', 'london')
dbutils.widgets.text('min_neighbourhood_prices', '10')

In [ ]:
IDENTIFIER = re.compile(r'[A-Za-z][A-Za-z0-9_]*')
CITY_KEY = re.compile(r'[a-z][a-z0-9_]*')

catalog = dbutils.widgets.get('catalog').strip()
silver_schema = dbutils.widgets.get('silver_schema').strip()
gold_schema = dbutils.widgets.get('gold_schema').strip()
training_cities = [c.strip().lower() for c in dbutils.widgets.get('training_cities').split(',') if c.strip()]
reference_city = dbutils.widgets.get('reference_city').strip().lower()
min_neighbourhood_prices = int(dbutils.widgets.get('min_neighbourhood_prices'))

for value in (catalog, silver_schema, gold_schema):
    if not IDENTIFIER.fullmatch(value):
        raise ValueError(f'Invalid identifier: {value!r}')
for value in training_cities + [reference_city]:
    if not CITY_KEY.fullmatch(value):
        raise ValueError(f'Invalid city key: {value!r}')
if not training_cities or reference_city not in training_cities:
    raise ValueError('reference_city must be one of training_cities')
if min_neighbourhood_prices < 1:
    raise ValueError('min_neighbourhood_prices must be positive')

S = f'{catalog}.{silver_schema}'
G = f'{catalog}.{gold_schema}'
TRAINING_SQL = ', '.join(f"'{c}'" for c in training_cities)
TRAINING_LABEL = ' + '.join(c.replace('_', ' ').title() for c in training_cities)
CITY_NAME = "initcap(replace(city, '_', ' '))"
print(f'silver={S}, gold={G}, training={training_cities}, reference={reference_city}')

## 2. Helpers and input checks

`create_view` writes a view with a table comment and a comment on every column. Genie uses these comments to interpret questions, and the column count is asserted so a comment cannot silently drift from its column.

In [ ]:
def sql_string(text):
    return "'" + text.replace('\\', '\\\\').replace("'", "\\'") + "'"


def create_view(name, comment, columns, select_sql):
    column_list = ',\n  '.join(f'`{col}` COMMENT {sql_string(desc)}' for col, desc in columns)
    spark.sql(f'CREATE OR REPLACE VIEW {G}.{name} (\n  {column_list}\n)\n'
              f'COMMENT {sql_string(comment)}\nAS {select_sql}')
    created = spark.table(f'{G}.{name}').columns
    assert created == [col for col, _ in columns], f'{name}: column mismatch {created}'
    print(f'Created {G}.{name} ({len(created)} columns)')


for table in (f'{S}.listings', f'{G}.city_metrics'):
    if not spark.catalog.tableExists(table):
        raise ValueError(f'Required ETL output is missing: {table}. Run the ETL job first.')

## 3. Listing-level view

`v_listings` is the single listing-level source for descriptive views. Invalid prices, availability values and review scores become NULL, so ordinary aggregates exclude them automatically.

In [ ]:
create_view('v_listings',
    'One row per listing snapshot from Silver with validated values only. Price is nightly advertised GBP. No host data.',
    [
        ('city', 'City key, lower case (e.g. london, bristol).'),
        ('city_name', 'Display name of the city.'),
        ('listing_id', 'Inside Airbnb listing ID; join key only.'),
        ('snapshot_date', 'Published Inside Airbnb snapshot date.'),
        ('price_gbp', 'Nightly advertised price in GBP; NULL when the source price is missing or invalid.'),
        ('has_valid_price', 'True when the listing has a valid positive advertised price.'),
        ('price_extreme', 'True when a valid price is below 10 or above 2000 GBP (review flag, not an error).'),
        ('room_type', 'Room type: Entire home/apt, Private room, Shared room or Hotel room.'),
        ('property_type', 'Host-declared property type.'),
        ('accommodates', 'Maximum number of guests.'),
        ('bedrooms', 'Number of bedrooms.'),
        ('beds', 'Number of beds.'),
        ('bathrooms', 'Number of bathrooms parsed from bathrooms text.'),
        ('neighbourhood', 'Neighbourhood (borough in London, ward in Bristol).'),
        ('latitude', 'Approximate latitude published by Inside Airbnb.'),
        ('longitude', 'Approximate longitude published by Inside Airbnb.'),
        ('minimum_nights', 'Minimum stay in nights.'),
        ('availability_30', 'Available calendar days in the next 30 days; not occupancy.'),
        ('availability_365', 'Available calendar days in the next 365 days; not occupancy.'),
        ('review_scores_rating', 'Overall review score 0-5; NULL when the listing has no rating.'),
    ],
    f'''
SELECT city, {CITY_NAME} AS city_name, listing_id, snapshot_date,
       CASE WHEN target_valid THEN price_gbp END AS price_gbp,
       COALESCE(target_valid, false) AS has_valid_price,
       COALESCE(price_extreme, false) AS price_extreme,
       room_type, property_type, accommodates, bedrooms, beds, bathrooms,
       neighbourhood_cleansed AS neighbourhood, latitude, longitude, minimum_nights,
       CASE WHEN NOT invalid_availability THEN availability_30 END AS availability_30,
       CASE WHEN NOT invalid_availability THEN availability_365 END AS availability_365,
       CASE WHEN NOT invalid_review_score THEN review_scores_rating END AS review_scores_rating
FROM {S}.listings''')

## 4. City overview

One row per city snapshot. `distinct_hosts` comes from the ETL table because host identifiers are not exposed in Gold.

In [ ]:
create_view('v_city_overview',
    'One row per city snapshot: listings, price, availability, reviews, hosts and room-type mix, each with its denominator.',
    [
        ('city', 'City key.'),
        ('city_name', 'Display name of the city.'),
        ('snapshot_date', 'Published snapshot date.'),
        ('total_listings', 'All listing snapshots that passed key checks.'),
        ('valid_price_listings', 'Listings with a valid advertised price; denominator of price statistics.'),
        ('price_coverage_pct', 'Percentage of listings with a valid price.'),
        ('median_price_gbp', 'Exact median nightly advertised price in GBP over valid prices.'),
        ('mean_price_gbp', 'Mean nightly advertised price in GBP over valid prices; sensitive to extreme values.'),
        ('extreme_price_listings', 'Valid prices below 10 or above 2000 GBP.'),
        ('availability_listings', 'Listings with valid availability; denominator of availability statistics.'),
        ('median_availability_365', 'Median available days in the next 365 days (not occupancy).'),
        ('avg_availability_365', 'Mean available days in the next 365 days (not occupancy).'),
        ('review_listings', 'Listings with an overall review score; denominator of review statistics.'),
        ('avg_review_score', 'Mean overall review score (0-5) among rated listings.'),
        ('distinct_hosts', 'Distinct hosts among valid listings, computed by the ETL.'),
        ('listings_per_host', 'Listings divided by distinct hosts.'),
        ('entire_home_share_pct', 'Percentage of listings that are Entire home/apt.'),
        ('neighbourhoods', 'Number of distinct neighbourhoods.'),
    ],
    f'''
WITH l AS (
  SELECT city, city_name, snapshot_date,
         COUNT(*) AS total_listings,
         COUNT(price_gbp) AS valid_price_listings,
         percentile(price_gbp, 0.5) AS median_price_gbp,
         AVG(price_gbp) AS mean_price_gbp,
         COUNT_IF(price_extreme) AS extreme_price_listings,
         COUNT(availability_365) AS availability_listings,
         percentile(availability_365, 0.5) AS median_availability_365,
         AVG(availability_365) AS avg_availability_365,
         COUNT(review_scores_rating) AS review_listings,
         AVG(review_scores_rating) AS avg_review_score,
         COUNT_IF(room_type = 'Entire home/apt') AS entire_home_listings,
         COUNT(DISTINCT neighbourhood) AS neighbourhoods
  FROM {G}.v_listings
  GROUP BY city, city_name, snapshot_date)
SELECT l.city, l.city_name, l.snapshot_date, l.total_listings, l.valid_price_listings,
       ROUND(100 * try_divide(l.valid_price_listings, l.total_listings), 2),
       ROUND(l.median_price_gbp, 2), ROUND(l.mean_price_gbp, 2), l.extreme_price_listings,
       l.availability_listings, ROUND(l.median_availability_365, 1), ROUND(l.avg_availability_365, 1),
       l.review_listings, ROUND(l.avg_review_score, 3), m.distinct_hosts,
       ROUND(try_divide(l.total_listings, m.distinct_hosts), 2),
       ROUND(100 * try_divide(l.entire_home_listings, l.total_listings), 2), l.neighbourhoods
FROM l LEFT JOIN {G}.city_metrics m
  ON m.city = l.city AND m.snapshot_date = l.snapshot_date''')

## 5. Segment views: room type, neighbourhood, guest capacity and price band

In [ ]:
create_view('v_room_type_mix',
    'One row per city snapshot and room type: listing share and median nightly GBP price.',
    [
        ('city', 'City key.'),
        ('city_name', 'Display name of the city.'),
        ('snapshot_date', 'Published snapshot date.'),
        ('room_type', 'Room type; Unknown when missing.'),
        ('listings', 'Listings of this room type.'),
        ('share_pct', 'Percentage of the city listings with this room type.'),
        ('valid_price_listings', 'Listings with a valid price; denominator of the median.'),
        ('median_price_gbp', 'Exact median nightly advertised price in GBP.'),
        ('avg_availability_365', 'Mean available days in the next 365 days (not occupancy).'),
    ],
    f'''
SELECT city, city_name, snapshot_date, COALESCE(room_type, 'Unknown') AS room_type,
       COUNT(*) AS listings,
       ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY city, snapshot_date), 2) AS share_pct,
       COUNT(price_gbp) AS valid_price_listings,
       ROUND(percentile(price_gbp, 0.5), 2) AS median_price_gbp,
       ROUND(AVG(availability_365), 1) AS avg_availability_365
FROM {G}.v_listings
GROUP BY city, city_name, snapshot_date, COALESCE(room_type, 'Unknown')''')

In [ ]:
create_view('v_neighbourhood_prices',
    f'One row per city snapshot and neighbourhood. Price ranks include only neighbourhoods with at least {min_neighbourhood_prices} valid prices.',
    [
        ('city', 'City key.'),
        ('city_name', 'Display name of the city.'),
        ('snapshot_date', 'Published snapshot date.'),
        ('neighbourhood', 'Neighbourhood; Unknown when missing.'),
        ('listings', 'Listings in the neighbourhood.'),
        ('valid_price_listings', 'Listings with a valid price; denominator of the median.'),
        ('median_price_gbp', 'Exact median nightly advertised price in GBP.'),
        ('entire_home_share_pct', 'Percentage of neighbourhood listings that are Entire home/apt.'),
        ('avg_availability_365', 'Mean available days in the next 365 days (not occupancy).'),
        ('avg_review_score', 'Mean overall review score (0-5) among rated listings.'),
        ('rankable', f'True when the neighbourhood has at least {min_neighbourhood_prices} valid prices.'),
        ('price_rank_in_city', '1 = highest median price within the city among rankable neighbourhoods; NULL otherwise.'),
        ('listings_rank_in_city', '1 = most listings within the city.'),
    ],
    f'''
WITH n AS (
  SELECT city, city_name, snapshot_date, COALESCE(neighbourhood, 'Unknown') AS neighbourhood,
         COUNT(*) AS listings, COUNT(price_gbp) AS valid_price_listings,
         percentile(price_gbp, 0.5) AS median_price_gbp,
         COUNT_IF(room_type = 'Entire home/apt') AS entire_home_listings,
         AVG(availability_365) AS avg_availability_365,
         AVG(review_scores_rating) AS avg_review_score
  FROM {G}.v_listings
  GROUP BY city, city_name, snapshot_date, COALESCE(neighbourhood, 'Unknown'))
SELECT city, city_name, snapshot_date, neighbourhood, listings, valid_price_listings,
       ROUND(median_price_gbp, 2), ROUND(100 * try_divide(entire_home_listings, listings), 2),
       ROUND(avg_availability_365, 1), ROUND(avg_review_score, 3),
       valid_price_listings >= {min_neighbourhood_prices},
       CASE WHEN valid_price_listings >= {min_neighbourhood_prices} THEN
         RANK() OVER (PARTITION BY city, snapshot_date, valid_price_listings >= {min_neighbourhood_prices}
                      ORDER BY median_price_gbp DESC) END,
       RANK() OVER (PARTITION BY city, snapshot_date ORDER BY listings DESC)
FROM n''')

In [ ]:
create_view('v_capacity_prices',
    'Median nightly GBP price by guest capacity (1 to 7 and 8+) per city snapshot.',
    [
        ('city', 'City key.'),
        ('city_name', 'Display name of the city.'),
        ('snapshot_date', 'Published snapshot date.'),
        ('guests', 'Guest capacity band: 1 to 7, or 8+.'),
        ('guests_order', 'Sort key of the guest band.'),
        ('listings', 'Listings in the band.'),
        ('valid_price_listings', 'Listings with a valid price; denominator of the median.'),
        ('median_price_gbp', 'Exact median nightly advertised price in GBP.'),
    ],
    f'''
SELECT city, city_name, snapshot_date,
       CASE WHEN accommodates >= 8 THEN '8+' ELSE CAST(accommodates AS STRING) END AS guests,
       LEAST(accommodates, 8) AS guests_order,
       COUNT(*), COUNT(price_gbp), ROUND(percentile(price_gbp, 0.5), 2)
FROM {G}.v_listings
WHERE accommodates >= 1
GROUP BY city, city_name, snapshot_date,
         CASE WHEN accommodates >= 8 THEN '8+' ELSE CAST(accommodates AS STRING) END,
         LEAST(accommodates, 8)''')

In [ ]:
create_view('v_price_bands',
    'Distribution of valid nightly advertised GBP prices in fixed bands per city snapshot.',
    [
        ('city', 'City key.'),
        ('city_name', 'Display name of the city.'),
        ('snapshot_date', 'Published snapshot date.'),
        ('price_band', 'Nightly price band in GBP.'),
        ('band_order', 'Sort key of the price band.'),
        ('listings', 'Listings with a valid price in the band.'),
        ('share_pct', 'Percentage of the city valid-price listings in the band.'),
    ],
    f'''
WITH b AS (
  SELECT city, city_name, snapshot_date,
         CASE WHEN price_gbp < 50 THEN 1 WHEN price_gbp < 100 THEN 2 WHEN price_gbp < 150 THEN 3
              WHEN price_gbp < 200 THEN 4 WHEN price_gbp < 300 THEN 5 WHEN price_gbp < 500 THEN 6
              WHEN price_gbp < 1000 THEN 7 ELSE 8 END AS band_order
  FROM {G}.v_listings WHERE price_gbp IS NOT NULL)
SELECT city, city_name, snapshot_date,
       element_at(array('under £50', '£50-99', '£100-149', '£150-199', '£200-299', '£300-499',
                        '£500-999', '£1,000+'), band_order),
       band_order, COUNT(*),
       ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY city, snapshot_date), 2)
FROM b GROUP BY city, city_name, snapshot_date, band_order''')

## 6. Cross-city comparison

Long format: one row per city and metric, with the difference from the reference city (city A, London by default). The dashboard pivots it into a side-by-side table.

In [ ]:
create_view('v_city_comparison',
    f'One row per city and headline metric, with the absolute and percentage difference from {reference_city.title()}.',
    [
        ('city', 'City key.'),
        ('city_name', 'Display name of the city.'),
        ('snapshot_date', 'Published snapshot date.'),
        ('metric_order', 'Display order of the metric.'),
        ('metric', 'Metric name.'),
        ('unit', 'Unit of the metric.'),
        ('value', 'Metric value for the city.'),
        ('reference_city', 'City used as the comparison reference.'),
        ('reference_value', 'Metric value of the reference city.'),
        ('difference_vs_reference', 'value minus reference_value.'),
        ('pct_difference_vs_reference', 'Percentage difference from the reference city.'),
    ],
    f'''
WITH m AS (
  SELECT city, city_name, snapshot_date, metric_order, metric, unit, value
  FROM {G}.v_city_overview
  LATERAL VIEW stack(10,
    1, 'Listings', 'count', CAST(total_listings AS DOUBLE),
    2, 'Listings with a valid price', 'count', CAST(valid_price_listings AS DOUBLE),
    3, 'Price coverage', '%', CAST(price_coverage_pct AS DOUBLE),
    4, 'Median nightly price', 'GBP', CAST(median_price_gbp AS DOUBLE),
    5, 'Mean nightly price', 'GBP', CAST(mean_price_gbp AS DOUBLE),
    6, 'Median available days (next 365)', 'days', CAST(median_availability_365 AS DOUBLE),
    7, 'Mean review score', '0-5', CAST(avg_review_score AS DOUBLE),
    8, 'Distinct hosts', 'count', CAST(distinct_hosts AS DOUBLE),
    9, 'Listings per host', 'ratio', CAST(listings_per_host AS DOUBLE),
    10, 'Entire-home share', '%', CAST(entire_home_share_pct AS DOUBLE)
  ) s AS metric_order, metric, unit, value),
r AS (SELECT metric_order, value AS reference_value FROM m WHERE city = '{reference_city}')
SELECT m.city, m.city_name, m.snapshot_date, m.metric_order, m.metric, m.unit, ROUND(m.value, 2),
       '{reference_city}', ROUND(r.reference_value, 2), ROUND(m.value - r.reference_value, 2),
       ROUND(100 * try_divide(m.value - r.reference_value, r.reference_value), 1)
FROM m LEFT JOIN r ON r.metric_order = m.metric_order''')

## 7. Model views

These views read `airbnb_gold.predictions`, written by `03_predict` (one row per listing snapshot, model version and run label). Required columns: city, listing_id, snapshot_date, model_version, split, actual_price_gbp, predicted_price_gbp and baseline_price_gbp. Optional columns (run_label, pipeline_version, baseline_level, prediction_status, generated_at) become NULL when absent.

- `run_label` separates runs: `development` for London and Bristol; `first_run`, `after_pipeline_fix` or `after_model_change` for the transfer city. Every metric is grouped by model version and run label, so the unchanged first run is never mixed with later runs.
- A prediction counts only when `prediction_status = 'valid'` and the value is a finite number; other predictions become NULL and reduce prediction coverage instead of distorting MAE.
- `evaluation_role` is `test` for the held-out split of training cities and `transfer` for every row of a non-training city (split `holdout`). Only these two roles are reported; train and inference rows are excluded.
- Combined MAE is computed over evaluated test listings of the training cities, so it is weighted by listing count rather than averaging city MAEs.

In [ ]:
predictions_table = f'{G}.predictions'
REQUIRED_PREDICTION_COLUMNS = ['city', 'listing_id', 'snapshot_date', 'model_version', 'split',
                               'actual_price_gbp', 'predicted_price_gbp', 'baseline_price_gbp']
OPTIONAL_PREDICTION_COLUMNS = ['run_label', 'pipeline_version', 'baseline_level', 'prediction_status', 'generated_at']

model_views_ready = spark.catalog.tableExists(predictions_table)
if model_views_ready:
    prediction_columns = spark.table(predictions_table).columns
    missing = [c for c in REQUIRED_PREDICTION_COLUMNS if c not in prediction_columns]
    if missing:
        model_views_ready = False
        print(f'SKIPPED model views: {predictions_table} lacks contract columns {missing}. Coordinate with the model owner.')
    opt = {c: (f'`{c}`' if c in prediction_columns else 'NULL') for c in OPTIONAL_PREDICTION_COLUMNS}
    print('Optional prediction columns present:', [c for c in OPTIONAL_PREDICTION_COLUMNS if c in prediction_columns])
else:
    print(f'SKIPPED model views: {predictions_table} does not exist yet. Rerun after the model writes predictions.')

In [ ]:
if model_views_ready:
    create_view('v_model_predictions',
        'One row per listing snapshot, model version and run label: actual, model and baseline prices with errors in GBP.',
        [
            ('city', 'City key.'),
            ('city_name', 'Display name of the city.'),
            ('listing_id', 'Inside Airbnb listing ID; join key only.'),
            ('snapshot_date', 'Published snapshot date.'),
            ('model_version', 'Model artifact version that produced the prediction (e.g. v1).'),
            ('run_label', 'Run: development (London, Bristol) or first_run, after_pipeline_fix, after_model_change (transfer city).'),
            ('pipeline_version', 'ETL pipeline version of the input row.'),
            ('split', 'Split from the model owner: train, test, inference (no valid price) or holdout (transfer city).'),
            ('evaluation_role', 'test = held-out rows of a training city; transfer = any row of a non-training city; otherwise the split name.'),
            ('in_evaluation_set', 'True for rows used in reported metrics (test or transfer).'),
            ('prediction_status', 'valid or invalid, written by the model owner.'),
            ('baseline_level', 'Baseline source: group (room type and guests), room_type or global fallback.'),
            ('generated_at', 'Time the prediction was generated.'),
            ('room_type', 'Room type of the listing.'),
            ('property_type', 'Property type of the listing.'),
            ('accommodates', 'Maximum number of guests.'),
            ('neighbourhood', 'Neighbourhood of the listing.'),
            ('actual_price_gbp', 'Actual nightly advertised price in GBP; NULL when the listing has no valid price.'),
            ('predicted_price_gbp', 'Model-predicted nightly price in GBP; NULL when the prediction is not valid.'),
            ('baseline_price_gbp', 'Baseline: training median price for the same room type and guest capacity, with predefined fallbacks.'),
            ('model_error_gbp', 'predicted minus actual in GBP; negative = model under-prices, positive = over-prices.'),
            ('model_abs_error_gbp', 'Absolute model error in GBP.'),
            ('baseline_abs_error_gbp', 'Absolute baseline error in GBP.'),
            ('model_pct_error', 'Model error as a percentage of the actual price.'),
            ('pricing_direction', 'under-priced, over-priced, exact or not evaluable.'),
            ('model_evaluable', 'True when actual price and a valid model prediction are both present.'),
            ('baseline_evaluable', 'True when actual and baseline prices are both present.'),
        ],
        f'''
WITH p AS (
  SELECT city, listing_id, snapshot_date, model_version, {opt['run_label']} AS run_label,
         {opt['pipeline_version']} AS pipeline_version, split,
         {opt['prediction_status']} AS prediction_status, {opt['baseline_level']} AS baseline_level,
         {opt['generated_at']} AS generated_at,
         CASE WHEN NOT isnan(actual_price_gbp) THEN actual_price_gbp END AS actual_price_gbp,
         CASE WHEN NOT isnan(predicted_price_gbp) AND COALESCE({opt['prediction_status']}, 'valid') = 'valid'
              THEN predicted_price_gbp END AS predicted_price_gbp,
         CASE WHEN NOT isnan(baseline_price_gbp) THEN baseline_price_gbp END AS baseline_price_gbp
  FROM {predictions_table})
SELECT p.city, initcap(replace(p.city, '_', ' ')), p.listing_id, p.snapshot_date, p.model_version,
       p.run_label, p.pipeline_version, p.split,
       CASE WHEN p.city NOT IN ({TRAINING_SQL}) THEN 'transfer'
            WHEN p.split = 'test' THEN 'test' ELSE COALESCE(p.split, 'unassigned') END,
       p.city NOT IN ({TRAINING_SQL}) OR p.split = 'test',
       p.prediction_status, p.baseline_level, p.generated_at,
       l.room_type, l.property_type, l.accommodates, l.neighbourhood,
       p.actual_price_gbp, p.predicted_price_gbp, p.baseline_price_gbp,
       ROUND(p.predicted_price_gbp - p.actual_price_gbp, 2),
       ROUND(ABS(p.predicted_price_gbp - p.actual_price_gbp), 2),
       ROUND(ABS(p.baseline_price_gbp - p.actual_price_gbp), 2),
       ROUND(100 * try_divide(p.predicted_price_gbp - p.actual_price_gbp, p.actual_price_gbp), 1),
       CASE WHEN p.actual_price_gbp IS NULL OR p.predicted_price_gbp IS NULL THEN 'not evaluable'
            WHEN p.predicted_price_gbp < p.actual_price_gbp THEN 'under-priced'
            WHEN p.predicted_price_gbp > p.actual_price_gbp THEN 'over-priced' ELSE 'exact' END,
       p.actual_price_gbp IS NOT NULL AND p.predicted_price_gbp IS NOT NULL,
       p.actual_price_gbp IS NOT NULL AND p.baseline_price_gbp IS NOT NULL
FROM p
LEFT JOIN {G}.v_listings l
  ON l.city = p.city AND l.listing_id = p.listing_id AND l.snapshot_date = p.snapshot_date''')

In [ ]:
RUN_KEYS = 'model_version, run_label, pipeline_version'
METRIC_AGGREGATES = '''
       COUNT(*) AS listings_in_set,
       COUNT(predicted_price_gbp) AS predicted_listings,
       ROUND(100 * try_divide(COUNT(predicted_price_gbp), COUNT(*)), 2) AS prediction_coverage_pct,
       COUNT_IF(model_evaluable) AS model_evaluated_listings,
       ROUND(AVG(model_abs_error_gbp), 2) AS model_mae_gbp,
       ROUND(percentile(model_abs_error_gbp, 0.5), 2) AS model_median_abs_error_gbp,
       COUNT_IF(baseline_evaluable) AS baseline_evaluated_listings,
       ROUND(AVG(baseline_abs_error_gbp), 2) AS baseline_mae_gbp,
       ROUND(percentile(baseline_abs_error_gbp, 0.5), 2) AS baseline_median_abs_error_gbp'''

if model_views_ready:
    create_view('v_model_performance',
        f'Model and baseline MAE in GBP per city, model version and run label on reported rows (test split of training cities, all rows of transfer cities), plus a combined {TRAINING_LABEL} row.',
        [
            ('city', "City key, or 'combined' for the training cities together."),
            ('city_name', 'Display name of the city or of the combined training cities.'),
            ('model_version', 'Model artifact version.'),
            ('run_label', 'development, or first_run / after_pipeline_fix / after_model_change for the transfer city.'),
            ('pipeline_version', 'ETL pipeline version.'),
            ('evaluation_role', 'test (training-city held-out rows) or transfer (unseen city).'),
            ('listings_in_set', 'Listings in the evaluated set, including rows without a price.'),
            ('predicted_listings', 'Listings with a valid model prediction.'),
            ('prediction_coverage_pct', 'Percentage of listings in the set with a valid prediction.'),
            ('model_evaluated_listings', 'Listings with both actual price and valid prediction; MAE denominator.'),
            ('model_mae_gbp', 'Model mean absolute error in GBP.'),
            ('model_median_abs_error_gbp', 'Model median absolute error in GBP; robust to extreme prices.'),
            ('baseline_evaluated_listings', 'Listings with both actual and baseline price; baseline MAE denominator.'),
            ('baseline_mae_gbp', 'Baseline mean absolute error in GBP.'),
            ('baseline_median_abs_error_gbp', 'Baseline median absolute error in GBP.'),
            ('mae_improvement_gbp', 'Baseline MAE minus model MAE; positive = model beats the baseline.'),
            ('mae_improvement_pct', 'MAE improvement as a percentage of baseline MAE.'),
        ],
        f'''
WITH e AS (SELECT * FROM {G}.v_model_predictions WHERE in_evaluation_set),
s AS (
  SELECT city, city_name, {RUN_KEYS}, evaluation_role, {METRIC_AGGREGATES}
  FROM e GROUP BY city, city_name, {RUN_KEYS}, evaluation_role
  UNION ALL
  SELECT 'combined', '{TRAINING_LABEL}', {RUN_KEYS}, 'test', {METRIC_AGGREGATES}
  FROM e WHERE city IN ({TRAINING_SQL})
  GROUP BY {RUN_KEYS})
SELECT *, ROUND(baseline_mae_gbp - model_mae_gbp, 2),
       ROUND(100 * try_divide(baseline_mae_gbp - model_mae_gbp, baseline_mae_gbp), 1)
FROM s''')

In [ ]:
SEGMENT_SELECT = '''
SELECT city, city_name, model_version, run_label, pipeline_version, evaluation_role,
       '{kind}' AS segment_type, COALESCE({column}, 'Unknown') AS segment,
       COUNT_IF(model_evaluable) AS evaluated_listings,
       ROUND(AVG(model_abs_error_gbp), 2) AS model_mae_gbp,
       ROUND(AVG(baseline_abs_error_gbp), 2) AS baseline_mae_gbp,
       ROUND(AVG(model_error_gbp), 2) AS mean_signed_error_gbp,
       ROUND(100 * try_divide(COUNT_IF(pricing_direction = 'under-priced'), COUNT_IF(model_evaluable)), 1) AS under_priced_share_pct,
       ROUND(percentile(actual_price_gbp, 0.5), 2) AS median_actual_price_gbp
FROM {source} WHERE in_evaluation_set AND model_evaluable
GROUP BY city, city_name, model_version, run_label, pipeline_version, evaluation_role, COALESCE({column}, 'Unknown')'''

if model_views_ready:
    source = f'{G}.v_model_predictions'
    create_view('v_model_error_segments',
        'Model and baseline error in GBP by room type and by neighbourhood on reported rows (test and transfer), per model version and run label.',
        [
            ('city', 'City key.'),
            ('city_name', 'Display name of the city.'),
            ('model_version', 'Model artifact version.'),
            ('run_label', 'development, or first_run / after_pipeline_fix / after_model_change for the transfer city.'),
            ('pipeline_version', 'ETL pipeline version.'),
            ('evaluation_role', 'test (training-city held-out rows) or transfer (unseen city).'),
            ('segment_type', 'room_type or neighbourhood.'),
            ('segment', 'Room type or neighbourhood name.'),
            ('evaluated_listings', 'Listings with actual price and valid prediction; denominator.'),
            ('model_mae_gbp', 'Model mean absolute error in GBP.'),
            ('baseline_mae_gbp', 'Baseline mean absolute error in GBP.'),
            ('mean_signed_error_gbp', 'Mean of predicted minus actual; negative = model under-prices the segment on average.'),
            ('under_priced_share_pct', 'Percentage of evaluated listings the model under-prices.'),
            ('median_actual_price_gbp', 'Median actual nightly price in GBP of evaluated listings.'),
        ],
        SEGMENT_SELECT.format(kind='room_type', column='room_type', source=source)
        + '\nUNION ALL\n'
        + SEGMENT_SELECT.format(kind='neighbourhood', column='neighbourhood', source=source))

## 8. Validation

The checks reconcile the views with the ETL outputs. They fail rather than publish inconsistent numbers.

In [ ]:
from pyspark.sql import functions as F

overview = spark.table(f'{G}.v_city_overview')
etl = spark.table(f'{G}.city_metrics').select('city', 'snapshot_date', F.col('total_listings').alias('etl_listings'),
                                              F.col('valid_price_listings').alias('etl_valid_prices'))
mismatch = (overview.join(etl, ['city', 'snapshot_date'], 'full')
            .filter('NOT (total_listings <=> etl_listings) OR NOT (valid_price_listings <=> etl_valid_prices)'))
assert mismatch.count() == 0, 'v_city_overview does not reconcile with city_metrics'

room_total = spark.table(f'{G}.v_room_type_mix').groupBy('city', 'snapshot_date').agg(F.sum('listings').alias('n'))
assert room_total.join(overview, ['city', 'snapshot_date']).filter('n != total_listings').count() == 0
assert spark.table(f'{G}.v_neighbourhood_prices').filter('rankable AND price_rank_in_city IS NULL').count() == 0

forbidden = ('host', 'url', 'name', 'description', 'price_quote', 'revenue', 'occupancy')
for view in ['v_listings', 'v_city_overview', 'v_room_type_mix', 'v_neighbourhood_prices']:
    leaked = [c for c in spark.table(f'{G}.{view}').columns
              if any(f in c for f in forbidden) and c not in ('city_name', 'distinct_hosts', 'listings_per_host')]
    assert not leaked, f'{view} exposes restricted columns {leaked}'

if model_views_ready:
    perf = spark.table(f'{G}.v_model_performance')
    assert perf.filter('model_evaluated_listings > listings_in_set').count() == 0
    display(perf.orderBy('model_version', 'run_label', 'city'))
    metrics_table = f'{G}.model_metrics'
    if spark.catalog.tableExists(metrics_table):
        # Reported by the model owner; MAE in v_model_performance must agree with these rows.
        display(spark.table(metrics_table).orderBy('model_version', 'dataset', 'city', 'predictor', 'metric'))

display(overview.orderBy('city'))
print('Validation passed. Model views created:', model_views_ready)

## 9. Third city

The views contain no city lists except the `training_cities` parameter, which only determines whether a city is reported as `test` or `transfer`. Do not run this notebook against third-city data until the release gate in `reports/city_c_protocol.md` is recorded. After the unchanged first run, rerun this notebook with unchanged parameters; the transfer city then appears on every dashboard page and in Genie.